In [1]:
# =========================================
# IMPORT LIBRARIES
# =========================================

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

import boto3
from botocore.client import Config

In [2]:
# =========================================
# INIT SPARK SESSION
# =========================================

spark = SparkSession.builder \
    .appName("SV3_Crypto_ETL") \
    .getOrCreate()

print("Spark Started Successfully")

# =========================================
# MINIO S3A CONFIG
# =========================================

hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()

hadoop_conf.set("fs.s3a.endpoint", "http://minio:9000")
hadoop_conf.set("fs.s3a.access.key", "admin")
hadoop_conf.set("fs.s3a.secret.key", "password123")
hadoop_conf.set("fs.s3a.path.style.access", "true")
hadoop_conf.set("fs.s3a.connection.ssl.enabled", "false")

hadoop_conf.set(
    "fs.s3a.aws.credentials.provider",
    "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider"
)

print("MinIO Configuration Completed")

Spark Started Successfully
MinIO Configuration Completed


In [3]:
# =========================================
# LOAD RAW DATA FROM MINIO
# =========================================

df = spark.read.csv(
    "s3a://crypto-raw-data/bitcoin_1m.csv",
    header=True,
    inferSchema=True
)

print("Raw Dataset Loaded")

print("Total Rows:", df.count())

df.show(20)

df.printSchema()

Raw Dataset Loaded
Total Rows: 1000
+-------------------+--------+--------+--------+--------+----------+
|          timestamp|    open|    high|     low|   close|    volume|
+-------------------+--------+--------+--------+--------+----------+
|2026-06-10 23:36:00|62311.01|62327.84|62256.19|62273.99|3.46491083|
|2026-06-10 23:37:00| 62274.7| 62342.4|62270.28|62300.86|1.27465983|
|2026-06-10 23:38:00|62300.54| 62314.0|62232.48|62292.79|1.28437369|
|2026-06-10 23:39:00|62292.78|62292.78|62226.36|62243.99|1.38524982|
|2026-06-10 23:40:00|62243.98|62303.25| 62236.0|62287.63|3.34283834|
|2026-06-10 23:41:00|62275.02|62291.01| 62201.8|62203.99|1.43590475|
|2026-06-10 23:42:00|62203.99|62219.88|62177.48|62205.29|1.91293337|
|2026-06-10 23:43:00| 62210.2| 62240.1|62191.89|62191.89|1.56146484|
|2026-06-10 23:44:00| 62191.9| 62208.0|62166.19|62166.19| 0.5891448|
|2026-06-10 23:45:00|62166.04|62166.04|62105.06|62118.01|1.92322625|
|2026-06-10 23:46:00|62112.55|62145.95|62025.94| 62042.4|5.38102484

In [4]:
# =========================================
# NULL CHECK
# =========================================

print("NULL CHECK")

df.select([
    F.count(
        F.when(F.col(c).isNull(), c)
    ).alias(c)
    for c in df.columns
]).show()

# =========================================
# DUPLICATE CHECK
# =========================================

total_rows = df.count()

unique_rows = df.dropDuplicates(
    ["timestamp"]
).count()

print("Total Rows:", total_rows)
print("Unique Rows:", unique_rows)
print("Duplicates:", total_rows - unique_rows)

NULL CHECK
+---------+----+----+---+-----+------+
|timestamp|open|high|low|close|volume|
+---------+----+----+---+-----+------+
|        0|   0|   0|  0|    0|     0|
+---------+----+----+---+-----+------+

Total Rows: 1000
Unique Rows: 1000
Duplicates: 0


In [5]:
# =========================================
# STANDARDIZE TIMESTAMP
# =========================================

df = df.withColumn("timestamp", F.to_timestamp("timestamp"))
df = df.orderBy("timestamp")

df.select(
    F.min("timestamp").alias("min_time"),
    F.max("timestamp").alias("max_time")
).show(truncate=False)

+-------------------+-------------------+
|min_time           |max_time           |
+-------------------+-------------------+
|2026-06-10 23:36:00|2026-06-11 16:15:00|
+-------------------+-------------------+



In [6]:
# =========================================
# GAP DETECTION
# =========================================

w = Window.orderBy("timestamp")

df = df.withColumn("prev_time", F.lag("timestamp").over(w))

df = df.withColumn(
    "diff_min",
    (F.unix_timestamp("timestamp")
     - F.unix_timestamp("prev_time")) / 60
)

df.select(
    "timestamp",
    "prev_time",
    "diff_min"
).show(20, False)

gap_count = df.filter(F.col("diff_min") > 1).count()

print("Gap Count:", gap_count)

+-------------------+-------------------+--------+
|timestamp          |prev_time          |diff_min|
+-------------------+-------------------+--------+
|2026-06-10 23:36:00|NULL               |NULL    |
|2026-06-10 23:37:00|2026-06-10 23:36:00|1.0     |
|2026-06-10 23:38:00|2026-06-10 23:37:00|1.0     |
|2026-06-10 23:39:00|2026-06-10 23:38:00|1.0     |
|2026-06-10 23:40:00|2026-06-10 23:39:00|1.0     |
|2026-06-10 23:41:00|2026-06-10 23:40:00|1.0     |
|2026-06-10 23:42:00|2026-06-10 23:41:00|1.0     |
|2026-06-10 23:43:00|2026-06-10 23:42:00|1.0     |
|2026-06-10 23:44:00|2026-06-10 23:43:00|1.0     |
|2026-06-10 23:45:00|2026-06-10 23:44:00|1.0     |
|2026-06-10 23:46:00|2026-06-10 23:45:00|1.0     |
|2026-06-10 23:47:00|2026-06-10 23:46:00|1.0     |
|2026-06-10 23:48:00|2026-06-10 23:47:00|1.0     |
|2026-06-10 23:49:00|2026-06-10 23:48:00|1.0     |
|2026-06-10 23:50:00|2026-06-10 23:49:00|1.0     |
|2026-06-10 23:51:00|2026-06-10 23:50:00|1.0     |
|2026-06-10 23:52:00|2026-06-10

In [7]:
# =========================================
# MA10 & MA60
# =========================================

w10 = Window.orderBy("timestamp").rowsBetween(-9, 0)
w60 = Window.orderBy("timestamp").rowsBetween(-59, 0)

df = df.withColumn("MA10", F.avg("close").over(w10))
df = df.withColumn("MA60", F.avg("close").over(w60))

df.select(
    "timestamp",
    "close",
    "MA10",
    "MA60"
).show(20, False)

+-------------------+--------+------------------+------------------+
|timestamp          |close   |MA10              |MA60              |
+-------------------+--------+------------------+------------------+
|2026-06-10 23:36:00|62273.99|62273.99          |62273.99          |
|2026-06-10 23:37:00|62300.86|62287.425         |62287.425         |
|2026-06-10 23:38:00|62292.79|62289.21333333334 |62289.21333333334 |
|2026-06-10 23:39:00|62243.99|62277.9075        |62277.9075        |
|2026-06-10 23:40:00|62287.63|62279.852         |62279.852         |
|2026-06-10 23:41:00|62203.99|62267.208333333336|62267.208333333336|
|2026-06-10 23:42:00|62205.29|62258.362857142856|62258.362857142856|
|2026-06-10 23:43:00|62191.89|62250.05375       |62250.05375       |
|2026-06-10 23:44:00|62166.19|62240.735555555555|62240.735555555555|
|2026-06-10 23:45:00|62118.01|62228.463         |62228.463         |
|2026-06-10 23:46:00|62042.4 |62205.304000000004|62211.54818181819 |
|2026-06-10 23:47:00|61927.4 |6216

In [8]:
# =========================================
# ROC + MOMENTUM
# =========================================

w = Window.orderBy("timestamp")

df = df.withColumn("close_lag10", F.lag("close", 10).over(w))

df = df.withColumn(
    "ROC",
    (F.col("close") - F.col("close_lag10"))
    / F.col("close_lag10") * 100
)

df = df.withColumn(
    "MOM",
    F.col("close") - F.col("close_lag10")
)

df.select(
    "close",
    "close_lag10",
    "ROC",
    "MOM"
).show(20, False)

+--------+-----------+--------------------+-------------------+
|close   |close_lag10|ROC                 |MOM                |
+--------+-----------+--------------------+-------------------+
|62273.99|NULL       |NULL                |NULL               |
|62300.86|NULL       |NULL                |NULL               |
|62292.79|NULL       |NULL                |NULL               |
|62243.99|NULL       |NULL                |NULL               |
|62287.63|NULL       |NULL                |NULL               |
|62203.99|NULL       |NULL                |NULL               |
|62205.29|NULL       |NULL                |NULL               |
|62191.89|NULL       |NULL                |NULL               |
|62166.19|NULL       |NULL                |NULL               |
|62118.01|NULL       |NULL                |NULL               |
|62042.4 |62273.99   |-0.3718888094371286 |-231.5899999999965 |
|61927.4 |62300.86   |-0.599445978755348  |-373.4599999999991 |
|61948.83|62292.79   |-0.552166631162288

In [9]:
# =========================================
# RSI 14
# =========================================

w1 = Window.orderBy("timestamp")
w14 = Window.orderBy("timestamp").rowsBetween(-13, 0)

df = df.withColumn("change", F.col("close") - F.lag("close").over(w1))

df = df.withColumn(
    "gain",
    F.when(F.col("change") > 0, F.col("change")).otherwise(0)
)

df = df.withColumn(
    "loss",
    F.when(F.col("change") < 0, -F.col("change")).otherwise(0)
)

df = df.withColumn("avg_gain", F.avg("gain").over(w14))
df = df.withColumn("avg_loss", F.avg("loss").over(w14))

df = df.withColumn(
    "RS",
    F.when(F.col("avg_loss") == 0, None)
     .otherwise(F.col("avg_gain") / F.col("avg_loss"))
)

df = df.withColumn(
    "RSI",
    F.when(F.col("avg_loss") == 0, 100)
     .when(F.col("avg_gain") == 0, 0)
     .otherwise(
         100 - (100 / (1 + F.col("RS")))
     )
)

print("RSI Created")

df.select(
    "timestamp",
    "close",
    "change",
    "gain",
    "loss",
    "avg_gain",
    "avg_loss",
    "RS",
    "RSI"
).show(20, False)

RSI Created
+-------------------+--------+-------------------+------------------+------------------+------------------+------------------+-------------------+------------------+
|timestamp          |close   |change             |gain              |loss              |avg_gain          |avg_loss          |RS                 |RSI               |
+-------------------+--------+-------------------+------------------+------------------+------------------+------------------+-------------------+------------------+
|2026-06-10 23:36:00|62273.99|NULL               |0.0               |0.0               |0.0               |0.0               |NULL               |100.0             |
|2026-06-10 23:37:00|62300.86|26.87000000000262  |26.87000000000262 |0.0               |13.43500000000131 |0.0               |NULL               |100.0             |
|2026-06-10 23:38:00|62292.79|-8.069999999999709 |0.0               |8.069999999999709 |8.95666666666754  |2.689999999999903 |3.3296158612148186 |76.903262736

In [10]:
# =========================================
# STOCHASTIC OSCILLATOR
# =========================================

df = df.withColumn(
    "highest_high",
    F.max("high").over(w14)
)

df = df.withColumn(
    "lowest_low",
    F.min("low").over(w14)
)

df = df.withColumn(
    "stoch_k",
    F.when(
        (F.col("highest_high") - F.col("lowest_low")) == 0,
        None
    ).otherwise(
        (F.col("close") - F.col("lowest_low"))
        /
        (F.col("highest_high") - F.col("lowest_low"))
        * 100
    )
)

w3 = Window.orderBy("timestamp").rowsBetween(-2, 0)

df = df.withColumn(
    "stoch_d",
    F.avg("stoch_k").over(w3)
)

print("Stochastic Created")

df.select(
    "timestamp",
    "close",
    "highest_high",
    "lowest_low",
    "stoch_k",
    "stoch_d"
).show(20, False)

Stochastic Created
+-------------------+--------+------------+----------+------------------+------------------+
|timestamp          |close   |highest_high|lowest_low|stoch_k           |stoch_d           |
+-------------------+--------+------------+----------+------------------+------------------+
|2026-06-10 23:36:00|62273.99|62327.84    |62256.19  |24.842986741098507|24.842986741098507|
|2026-06-10 23:37:00|62300.86|62342.4     |62256.19  |51.815334647951175|38.329160694524845|
|2026-06-10 23:38:00|62292.79|62342.4     |62232.48  |54.86717612809191 |43.841832505713874|
|2026-06-10 23:39:00|62243.99|62342.4     |62226.36  |15.193036883830791|40.62518255329129 |
|2026-06-10 23:40:00|62287.63|62342.4     |62226.36  |52.800758359183334|40.95365712370201 |
|2026-06-10 23:41:00|62203.99|62342.4     |62201.8   |1.5576102418172653|23.183801828277126|
|2026-06-10 23:42:00|62205.29|62342.4     |62177.48  |16.86272131942637 |23.74036330680899 |
|2026-06-10 23:43:00|62191.89|62342.4     |62177.48

In [11]:
# =========================================
# BUY / SELL LABEL
# =========================================

df = df.withColumn(
    "label",
    F.when(F.col("MA10") > F.col("MA60"), 1).otherwise(0)
)

print("Buy/Sell Label Created")

print("Label Distribution")

df.groupBy("label").count().show()

df.select(
    "timestamp",
    "MA10",
    "MA60",
    "label"
).show(20, False)

Buy/Sell Label Created
Label Distribution
+-----+-----+
|label|count|
+-----+-----+
|    0|  479|
|    1|  521|
+-----+-----+

+-------------------+------------------+------------------+-----+
|timestamp          |MA10              |MA60              |label|
+-------------------+------------------+------------------+-----+
|2026-06-10 23:36:00|62273.99          |62273.99          |0    |
|2026-06-10 23:37:00|62287.425         |62287.425         |0    |
|2026-06-10 23:38:00|62289.21333333334 |62289.21333333334 |0    |
|2026-06-10 23:39:00|62277.9075        |62277.9075        |0    |
|2026-06-10 23:40:00|62279.852         |62279.852         |0    |
|2026-06-10 23:41:00|62267.208333333336|62267.208333333336|0    |
|2026-06-10 23:42:00|62258.362857142856|62258.362857142856|0    |
|2026-06-10 23:43:00|62250.05375       |62250.05375       |0    |
|2026-06-10 23:44:00|62240.735555555555|62240.735555555555|0    |
|2026-06-10 23:45:00|62228.463         |62228.463         |0    |
|2026-06-10 23:

In [12]:
# =========================================
# FEATURE TABLE
# =========================================

final_df = df.select(
    "timestamp",
    "open", "high", "low", "close", "volume",
    "MA10", "MA60",
    "ROC", "MOM",
    "RSI",
    "stoch_k", "stoch_d",
    "label"
)

print("Rows Before DropNA:", final_df.count())

final_df = final_df.dropna()

print("Rows After DropNA:", final_df.count())

Rows Before DropNA: 1000
Rows After DropNA: 990


In [13]:
# =========================================
# CREATE MINIO BUCKET
# =========================================

s3 = boto3.client(
    "s3",
    endpoint_url="http://minio:9000",
    aws_access_key_id="admin",
    aws_secret_access_key="password123",
    config=Config(signature_version="s3v4")
)

bucket_name = "crypto-feature-table"

if bucket_name not in [
    b["Name"]
    for b in s3.list_buckets()["Buckets"]
]:
    s3.create_bucket(
        Bucket=bucket_name
    )
    print("Bucket Created")
else:
    print("Bucket Already Exists")

Bucket Already Exists


In [14]:
# =========================================
# SAVE FEATURE TABLE
# =========================================

final_df.write \
    .mode("overwrite") \
    .parquet(
        "s3a://crypto-feature-table/features/"
    )

print("Feature Table Saved")

Feature Table Saved


In [15]:
# =========================================
# VERIFY OUTPUT
# =========================================

verify_df = spark.read.parquet(
    "s3a://crypto-feature-table/features/"
)

print(
    "Rows Written:",
    verify_df.count()
)

verify_df.show(30, False)

verify_df.printSchema()

Rows Written: 990
+-------------------+--------+--------+--------+--------+-----------+------------------+------------------+---------------------+-------------------+------------------+------------------+------------------+-----+
|timestamp          |open    |high    |low     |close   |volume     |MA10              |MA60              |ROC                  |MOM                |RSI               |stoch_k           |stoch_d           |label|
+-------------------+--------+--------+--------+--------+-----------+------------------+------------------+---------------------+-------------------+------------------+------------------+------------------+-----+
|2026-06-10 23:46:00|62112.55|62145.95|62025.94|62042.4 |5.38102484 |62205.304000000004|62211.54818181819 |-0.3718888094371286  |-231.5899999999965 |19.138615708537543|5.201289262465769 |3.5525322231876095|0    |
|2026-06-10 23:47:00|62041.65|62058.01|61927.4 |61927.4 |4.41341656 |62167.958000000006|62187.86916666667 |-0.599445978755348   |-

In [16]:
# df = spark.read.parquet(
#     "s3a://crypto-feature-table/features/"
# )

In [17]:
# df.printSchema()

In [18]:
# df.show(20, truncate=False)